<a href="https://colab.research.google.com/github/Kauustubbh/Skin-Lesion/blob/main/notebooks/02_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive, userdata

# 1. Ensure Drive is mounted
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/SkinLesionClassifier'

# 2. Load token from Colab Secrets
try:
    token = userdata.get('Kaggle_api_key')
except:
    token = userdata.get('KAGGLE_API_TOKEN')

# Set environment variable for Kaggle CLI
os.environ['KAGGLE_API_TOKEN'] = token

# 3. Download and unzip dataset
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 --unzip -p data/
print("✅ Dataset downloaded and unzipped successfully.")

In [ ]:
import pandas as pd
import glob

# 1. Read split contract exported by Notebook 01 from Drive
split_csv_path = os.path.join(PROJECT_DIR, 'train_val_test_split.csv')
df = pd.read_csv(split_csv_path)

# 2. Resolve image paths across both downloaded subfolders
img_paths = glob.glob('data/HAM10000_images_part_1/*.jpg') + glob.glob('data/HAM10000_images_part_2/*.jpg')
img_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in img_paths}

df['path'] = df['image_id'].map(img_lookup)

# 3. Encoded diagnostic class shortcodes to 0-6 integer labels
label_mapping = {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}
df['label'] = df['dx'].map(label_mapping)

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)

print(f"✅ Split contract loaded successfully: {len(train_df)} train samples, {len(val_df)} validation samples.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Geometric augmentations ONLY (preserves clinical skin lesion color features)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = row['label']
        if self.transform:
            image = self.transform(image)
        return image, label

BATCH_SIZE = 64  # Increased batch size for fast T4 GPU processing

train_loader = DataLoader(
    SkinDataset(train_df, train_transforms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    SkinDataset(val_df, val_transforms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Compute inverse frequency class weights to fix dataset imbalance
class_counts = train_df['label'].value_counts().sort_index().values
total_samples = len(train_df)
num_classes = len(class_counts)

class_weights = total_samples / (num_classes * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_weights_tensor = class_weights_tensor.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
print("✅ Datasets and fast DataLoaders ready. Device set to:", device)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights

num_classes = 7

model = resnet50(weights=ResNet50_Weights.DEFAULT)

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

optimizer_stage1 = optim.Adam(model.fc.parameters(), lr=1e-3)

print("🚀 Starting Stage 1: FC Head Warmup (3 Epochs)...")
for epoch in range(3):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer_stage1.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_stage1.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/3 - Loss: {epoch_loss:.4f}")

for param in model.layer4.parameters():
    param.requires_grad = True

optimizer_stage2 = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}
])

print("\n🚀 Starting Stage 2: Fine-Tuning Layer 4 & FC Head...")
num_epochs_stage2 = 10

for epoch in range(num_epochs_stage2):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer_stage2.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_stage2.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs_stage2} - Loss: {epoch_loss:.4f}")

checkpoint_path = os.path.join(PROJECT_DIR, 'baseline_resnet50.pth')
torch.save(model.state_dict(), checkpoint_path)
print(f"\n🎉 Training finished! Checkpoint saved to: {checkpoint_path}")